# Création GastronoBot

Packages & Data 

In [1]:
import torch

import os
import requests
import fitz
from tqdm.auto import tqdm
################# 
#import pdf
pdf_path = "intern_recipe.pdf"

#Download it 
if not os.path.exists(pdf_path):
    print(f"Downloading {pdf_path}...")
    url = "https://pdf.infobooks.org/ING/Raiz%20Cocina%20y%20Bebidas/My-favourite-recipe-International-Cookbook-University-of-G-ttingen.pdf"
    
    response = requests.get(url)
    if response.status_code == 200:
        with open(pdf_path, 'wb') as f:
            f.write(response.content)
        print("Download successful!")
    else:
        print(f"Failed to download the PDF. Status code: {response.status_code}")

else :
    print(f"File {pdf_path} already exists. Skipping download.")
    
# Nous avons le chemin vers le pdf maintenant chargeons le pdf
def text_formatter(text:str) -> str:
    #on retire les sauts de ligne
    cleaned_text = text.replace('\n', ' ').strip()
    
    return cleaned_text

def open_and_read_pdf(pdf_path:str) -> list:
    # Ouvrir le PDF
    doc = fitz.open(pdf_path)
    
    # Extraire le texte de chaque page
    pages_and_texts = []
    for page_number, page in enumerate(tqdm(doc)):
        text = page.get_text()
        text = text_formatter(text = text)
        pages_and_texts.append({"page_number": page_number + 1, 
                                 "page_char_count": len(text), 
                                 "page_word_count": len(text.split(" ")),
                                 "page_sentnece_count": len(text.split(".")),
                                 "page_token_count": len(text) / 4, #en moyenne un token correspond à 4 caractères en anglais
                                 "text": text})
    
    return pages_and_texts
    
pages_and_texts = open_and_read_pdf(pdf_path = pdf_path)
pages_and_texts[:2]

c:\Users\ugoas\Desktop\Stage Geeng\GastronoBot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


File intern_recipe.pdf already exists. Skipping download.


100%|██████████| 150/150 [00:00<00:00, 941.94it/s]


[{'page_number': 1,
  'page_char_count': 117,
  'page_word_count': 21,
  'page_sentnece_count': 1,
  'page_token_count': 29.25,
  'text': 'Favourite recipes from all over the world  – easy to be prepared in Germany My favourite recipes My favourite recipes'},
 {'page_number': 2,
  'page_char_count': 1180,
  'page_word_count': 188,
  'page_sentnece_count': 8,
  'page_token_count': 295.0,
  'text': 'PREFACE My favourite recipe The University of Göttingen is characterised by  a particularly ”international spirit”, which is  reflected in various ways. We are involved in cooperative projects with  universities in 107 countries in Europe, Asia,  North and South America, as well as in Africa,  and are active in international networks such as  the U4 network, The Guild of European Research  Intensive Universities, the German-Japanese  HeKKSaGOn network and the Coimbra Group.  We lend strong support to students and resear\xad chers by providing numerous scholarships for  international exchange.

## Document preprocess & création embedding

! Objectif traiter le pdf pour avoir des chunks pour créer l'embedding pour la recherche vectorielle 

In [3]:
#Exemple de page et de texte extrait du PDF.
import random
random.sample(pages_and_texts, k = 2)

[{'page_number': 39,
  'page_char_count': 1735,
  'page_word_count': 345,
  'page_sentnece_count': 24,
  'page_token_count': 433.75,
  'text': 'SIMPLE STEPS of how to prepare it \t 1|\t\x07Chop one onion quite finely.  \t 2|\t\x07Trim the meat. Put some oil or fat in the  bottom of a reasonably-large sized saucepan  or heat resistant/metal casserole dish.  \t 3|\t\x07Heat the oil, then add the meat and onion.  Cook, stirring occasionally, on a medium  heat, until the meat is brown on all sides and  the onions start to cook through.  \t 4|\t\x07Add enough water to well cover the meat and  onions (roughly 800ml) and 3 stock cubes.  Bring to the boil.  \t 5|\t\x07Cover the saucepan with a lid, lower the heat  to a slow boil, and let it stew for roughly one  hour. The main thing is that the meat is  cooked through and at least starting to  tenderise. The longer and slower the cooking,  the better.  \t 6|\t\x07In the meantime, start to prepare the other  vegetables. Peel your selection of r

In [2]:
import pandas as pd
df_pages = pd.DataFrame(pages_and_texts)
df_pages.head()
df_pages.describe()

,page_number,page_char_count,page_word_count,page_sentnece_count,page_token_count
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,75.500000,869.506667,162.386667,8.480000,217.376667
std,43.445368,386.301370,75.685218,6.639034,96.575342
min,1.000000,3.000000,1.000000,1.000000,0.750000
25%,38.250000,601.750000,112.000000,3.000000,150.437500
50%,75.500000,826.500000,152.500000,7.000000,206.625000
75%,112.750000,1070.000000,203.000000,11.000000,267.500000
max,150.000000,2200.000000,440.000000,32.000000,550.000000


Pourquoi est ce important de prendre en compte les tokens ?
1. c'est l'unité de mesure des LLM et des Embeddings. Ils n'ont pas une quantité infinie.

### On va séparer les pages en phrases

On va chercher à grouper nos phrases plus par page mais recette. Ce qui est intéressant c'est que dans ce pdf,ce qu'a partir de la page 8 deux pages sont consacrés à une personne et la recette qu'elle présente.

In [15]:
#On va à partir de la page 8 groupe et créer un json de ce livre avec le debut normal de page 1 à 7 et ensuite à partir de la page 8 on va grouper les pages par 2 pour avoir les recettes et les gens associés

def json_pour_embedding(pages_and_texts : list) -> list:
    json_for_embedding = []
    
    #on ajoute les 7 premières pages
    for page in pages_and_texts[:7]:
        json_for_embedding.append({"page_number": page["page_number"], 
                                   "text": page["text"]})
    
    #Ensuite on groupe les pages par 2 à partir de la page 8
    for i in range(7, len(pages_and_texts), 2):
        group_pages = pages_and_texts[i:i+2]
        group_text = " ".join([page["text"] for page in group_pages])
        json_for_embedding.append({"page_number": f"{group_pages[0]['page_number']}-{group_pages[-1]['page_number']}", 
                                   "text": group_text})
    
    #on va ajouter le nomber de token par text pour pouvoir faire le lien avec les limites de token des modèles d'embedding
    for item in json_for_embedding:
        item["token_count"] = len(item["text"]) / 4 #en moyenne un token correspond à 4 caractères en anglais
        
    return json_for_embedding

recipes = json_pour_embedding(pages_and_texts = pages_and_texts)
random.sample(recipes, 2)

df_recipes = pd.DataFrame(recipes)
df_recipes.head()


,page_number,text,token_count
0,1,Favourite recipes from all over the world – e...,29.25
1,2,PREFACE My favourite recipe The University of ...,295.00
2,3,PREFACE The number of international students a...,267.75
3,4,CORPORATE HEALTH MANAGEMENT Georg-August-Uni...,228.50
4,5,INTERNATIONAL OFFICE Georg-August-Universität...,258.50


### Création de notre embedding pour notre database

On va chercher à transformer nos groupements de recettes en nombres, donnant des embeddings. 
Pour le choix du modèle, il a été choisi un modèle simple et rapide sur CPU pour continuer = "paraphrase-multilingual-mpnet-base-v2"

In [16]:
from sentence_transformers import SentenceTransformer
embeddings_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20567.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


création de l'espace vectoriel

In [17]:
import numpy as np

texts = [r["text"] for r in recipes]
texts[1]

embeddings_model.encode(texts[1], show_progress_bar=True)

embeddings_recette = embeddings_model.encode(texts,
                                  batch_size= 32,
                                  show_progress_bar=True)

print(f"Shape des embeddings : {embeddings_recette.shape}")



Batches: 100%|██████████| 3/3 [00:01<00:00,  1.77it/s]

Shape des embeddings : (79, 384)


In [18]:
embeddings_recette[0][:10]

#On va ajouter la colonne des embeddings à notre dataframe
df_recipes["embedding"] = list(embeddings_recette)
df_recipes.head()

,page_number,text,token_count,embedding
0,1,Favourite recipes from all over the world – e...,29.25,"[-0.03986102, -0.012648857, 0.029558245, 0.048..."
1,2,PREFACE My favourite recipe The University of ...,295.00,"[0.016805885, -0.027275806, 0.03696693, -0.051..."
2,3,PREFACE The number of international students a...,267.75,"[-0.007708075, 0.024778374, 0.03423958, 0.0178..."
3,4,CORPORATE HEALTH MANAGEMENT Georg-August-Uni...,228.50,"[-0.031611025, 0.019724159, 0.019372849, -0.02..."
4,5,INTERNATIONAL OFFICE Georg-August-Universität...,258.50,"[-0.070061594, -0.05560431, 0.09010976, -0.043..."


On va stocker dorénavant cet embedding 

In [19]:
recette_embed_df = pd.DataFrame(df_recipes)
embed = "intern_recipe_df.csv"
recette_embed_df.to_csv(embed, index=False)

Pour +100K chunks il faut passer à un vector database